<a href="https://colab.research.google.com/github/demekeendalie/multi-emotion-classification/blob/main/mBART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import MBartTokenizer, MBartForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
# Try reading the CSV with different encodings
data = pd.read_excel('/content/drive/MyDrive/preprocessed emotion_dataset.xlsx')
display(data.tail(10))

,comments,sad,happy,anger,fear,disgust,surprise,contempt,neutral
22020,አሁንም ዐቢይ አህመድ ኢትዮጵያን ቅስሟን ሰብሬ እጥላለው እያለ እየታገለ ...,0,0,1,1,1,0,0,0
22021,98 የወለጋ የተፈናቀለው የተገደለው ሙስሊም አማራ ነው ኦሮሞ እና ሙስሊም...,0,0,1,0,1,1,0,0
22022,ጠላት ደነገጠ በባህር ኃይላችን ፐ ፐ ፐ ባህር ኃይል ችግኝ ቤተመንግስት ...,1,0,0,0,0,1,0,0
22023,በዘር ጥላቻ ተወጥረህ መቸም አይገባህም ስለአገር ልክልክህን ነግረውሃ ረጅ...,0,0,1,0,1,0,0,0
22024,እርፍ በል ሽሜ ሊያውም ሙሉ ብሔራዊ ቡድኑን ነው ያደመከው። የሚገርም ህብ...,0,0,1,0,0,0,0,0
22025,ለኔ የምንግዜም ጀግና ሁሌም አምንህ ነበር ሁሉም በአብይ ፍቅር ባበደበት ...,1,0,0,0,0,1,0,0
22026,የአብይ ጥበቃ አራዊት ከብልፅግና በላይ ፅንፈኛ ከብልፅግና በላይ የኢትዮጵ...,0,0,1,0,1,0,0,0
22027,አለምነህ ዋሴ ምርጥ ይሁዳዊ ጋዜጠኛ በርታልን እኔ የኛ ሰው መሆንክን ባወ...,1,0,0,0,0,1,0,0
22028,ሰው ሁሉ እንደ እርስዎ አላዋቂ፣ ግብዝ እና ተግባር በሌለው ተንኮልን ባነ...,0,0,1,0,1,0,0,0
22029,ሶፎንያስ በሕይወት ሲኖር ያማልዳል !!!ከሞት ቦኃላ ወደ እንጦርጦስ ስለሆ...,0,0,0,0,0,0,0,0


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Extract label columns from the DataFrame
label_columns = ['sad', 'happy', 'anger', 'fear', 'disgust', 'surprise', 'contempt', 'neutral']
labels = data[label_columns]

# The labels are already in binary format, so we can directly use their values.
binary_labels = labels.values

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

# First split: data and corresponding binary_labels
train_val_df, test_dataset, train_val_labels, test_labels = train_test_split(
    data, binary_labels, test_size=0.10, random_state=42
)

# Second split: train_val_df and train_val_labels
train_dataset, evaluation_dataset, train_labels, val_labels = train_test_split(
    train_val_df, train_val_labels, test_size=0.10, random_state=42
)

print('Training dataset shape: ', train_dataset.shape)
print('Validation dataset shape: ', evaluation_dataset.shape)
print('Testing dataset shape: ', test_dataset.shape)

Training dataset shape:  (17844, 9)
Validation dataset shape:  (1983, 9)
Testing dataset shape:  (2203, 9)


In [ ]:
# Initialize tokenizer
tokenizer = MBartTokenizer.from_pretrained("facebook/mbart-large-50")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

In [ ]:
# Extract texts from the datasets
train_texts = train_dataset['comments'].tolist()
val_texts = evaluation_dataset['comments'].tolist()
test_texts = test_dataset['comments'].tolist()

# Tokenize the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
test_encodings=tokenizer(test_texts, truncation=True, padding=True, max_length=512)

In [ ]:
# Create PyTorch Dataset
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)  # Use float for multi-label
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = MultiLabelDataset(train_encodings, train_labels)
val_dataset = MultiLabelDataset(val_encodings, val_labels)
test_dataset = MultiLabelDataset(test_encodings, test_labels)

In [ ]:
# Load model
model = MBartForSequenceClassification.from_pretrained(
    "facebook/mbart-large-50",
    num_labels=len(label_columns),
    problem_type="multi_label_classification"  # Set problem type
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MBartForSequenceClassification LOAD REPORT from: facebook/mbart-large-50
Key                                 | Status     | 
------------------------------------+------------+-
final_logits_bias                   | UNEXPECTED | 
lm_head.weight                      | UNEXPECTED | 
classification_head.dense.bias      | MISSING    | 
classification_head.out_proj.weight | MISSING    | 
classification_head.dense.weight    | MISSING    | 
classif

In [ ]:
# Set training arguments
training_args = TrainingArguments(
   output_dir="./multi_emotion",
   eval_strategy='epoch',
   save_strategy='epoch',
   logging_strategy='epoch',
   num_train_epochs=5,
   learning_rate=1e-6,
   per_device_train_batch_size=4,  # batch size per device during training
   per_device_eval_batch_size=4,   # batch size for evaluation
   warmup_steps=1000,                # number of warmup steps for learning rate
   weight_decay=0.01,
   logging_dir='./logs',            # directory for storing logs (Corrected parameter name)
   logging_steps=20,
   report_to="none",
   load_best_model_at_end= True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    hamming_loss
)
def custom_metrics(eval_pred):
    predictions_output = eval_pred.predictions
    labels = eval_pred.label_ids

    # Extract logits from predictions_output. If it's a tuple, assume logits are the first element.
    if isinstance(predictions_output, tuple):
        logits = predictions_output[0]
    else:
        logits = predictions_output

    # Apply sigmoid to logits to get probabilities for each label
    probabilities = torch.sigmoid(torch.from_numpy(logits)).numpy()
    # Convert probabilities to binary predictions using a threshold (e.g., 0.5)
    predictions_binary = (probabilities >= 0.5).astype(int) # Renamed to avoid conflict

    # Ensure labels are integers (they were floats from binary_labels.values)
    references = labels.astype(int)

    # Compute metrics using sklearn.metrics, which explicitly supports multi-label
    f1 = f1_score(references, predictions_binary, average="micro")
    precision = precision_score(references, predictions_binary, average="micro")
    recall = recall_score(references, predictions_binary, average="micro")
    accuracy = accuracy_score(references, predictions_binary)
    hamming = hamming_loss(references, predictions_binary)

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "hamming_loss": hamming
    }

In [ ]:
# Trainer
from transformers import EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=custom_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.232317,0.243868,0.862552,0.912274,0.817969,0.537065,0.098399,35.519400,55.829000,13.964000
2,0.221820,0.248099,0.863370,0.923677,0.810454,0.538074,0.096823,31.758000,62.441000,15.618000
3,0.210798,0.247753,0.865986,0.889868,0.843353,0.549168,0.098525,31.784700,62.388000,15.605000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13383, training_loss=0.22164477117658596, metrics={'train_runtime': 2832.3962, 'train_samples_per_second': 31.5, 'train_steps_per_second': 7.875, 'total_flos': 2.908967545007309e+16, 'train_loss': 0.22164477117658596, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

{'eval_loss': 0.2573304772377014, 'eval_f1': 0.8504446240905417, 'eval_precision': 0.9072093825457054, 'eval_recall': 0.8003651856360317, 'eval_accuracy': 0.5124829777576033, 'eval_hamming_loss': 0.10497049477984567, 'eval_runtime': 37.6517, 'eval_samples_per_second': 58.51, 'eval_steps_per_second': 14.634, 'epoch': 3.0}


In [ ]:
print("Evaluation metrics:")
for key, value in test_results.items():
    print(f"{key}: {value}")

Evaluation metrics:
eval_loss: 0.2573304772377014
eval_f1: 0.8504446240905417
eval_precision: 0.9072093825457054
eval_recall: 0.8003651856360317
eval_accuracy: 0.5124829777576033
eval_hamming_loss: 0.10497049477984567
eval_runtime: 37.6517
eval_samples_per_second: 58.51
eval_steps_per_second: 14.634
epoch: 3.0
